In [ ]:
!pip install pot

In [ ]:
import sys;
import torch;
import torch.nn as nn;
import torch.optim as optim
from torch.nn import functional as F
from torch.autograd.functional import jacobian
from torch.autograd import Variable
import numpy as np;
import matplotlib.pyplot as plt;
from matplotlib.pyplot import figure
from matplotlib.colors import LinearSegmentedColormap
import math
import os
import seaborn as sns
import ot
import random
from functorch import vmap
from functorch import jacfwd
from random import randint
from pathlib import Path
from mpl_toolkits.axes_grid1 import AxesGrid

print("Packages:");
print("torch.__version__ = " + str(torch.__version__));
print("numpy.__version__ = " + str(np.__version__));
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
class Net_trunk(nn.Module):
    def __init__(self):
        super(Net_trunk, self).__init__()
        self.hidden_layer1 = nn.Linear(2,50,bias=True)
        self.hidden_layer2 = nn.Linear(50,100, bias=True);  self.hidden_layer3 = nn.Linear(100,100, bias=True)
        self.hidden_layer4 = nn.Linear(100,100, bias=True);  self.hidden_layer5 = nn.Linear(100,200, bias=True)
        self.hidden_layer6 = nn.Linear(200,200, bias=True);  self.hidden_layer7 = nn.Linear(200,300, bias=True)
        self.output_layer = nn.Linear(300,200, bias=True)

    def forward(self, x,t):
        inputs = torch.cat([x,t],1) 
        layer1_out = F.gelu(self.hidden_layer1(inputs)); layer2_out = F.gelu(self.hidden_layer2(layer1_out))
        layer3_out = F.gelu(self.hidden_layer3(layer2_out));  layer4_out = F.gelu(self.hidden_layer4(layer3_out))
        layer5_out = F.gelu(self.hidden_layer5(layer4_out));  layer6_out = F.gelu(self.hidden_layer6(layer5_out))
        layer7_out = F.gelu(self.hidden_layer7(layer6_out))
        output = self.output_layer(layer7_out)
        return output
    
    def compute_u_t(self, x, t):
        self.u_t = torch.autograd.functional.jacobian(self, (x,t), create_graph=True)
        return self.u_t[1]
    
    def compute_u_x(self, x, t):
        self.u_x = torch.autograd.functional.jacobian(self, (x,t), create_graph=True)
        return self.u_x[0]

In [ ]:
class Net_branch(nn.Module):
    def __init__(self):
        super(Net_branch, self).__init__()
        self.hidden_layer1 = nn.Linear(50,100, bias=True); self.hidden_layer2 = nn.Linear(100,100, bias=True)
        self.hidden_layer3 = nn.Linear(100,100, bias=True); self.hidden_layer4 = nn.Linear(100,100, bias=True)
        self.hidden_layer5 = nn.Linear(100,150, bias=True); self.hidden_layer6 = nn.Linear(150,200, bias=True)
        self.output_layer = nn.Linear(200,200, bias=True)

    def forward(self, x):
        inputs = torch.cat([x],1) 
        layer1_out = F.gelu(self.hidden_layer1(inputs)); layer2_out = F.gelu(self.hidden_layer2(layer1_out))
        layer3_out = F.gelu(self.hidden_layer3(layer2_out));  layer4_out = F.gelu(self.hidden_layer4(layer3_out))
        layer5_out = F.gelu(self.hidden_layer5(layer4_out));  layer6_out = F.gelu(self.hidden_layer6(layer5_out))
        output = self.output_layer(layer6_out)
        return output

In [ ]:
HJ_branch0 = Net_branch()
HJ_branch1 = Net_branch()
HJ_branch0.to(device)
HJ_branch1.to(device)
HJ_trunk = Net_trunk()
HJ_trunk.to(device)

Cty_branch0 = Net_branch()
Cty_branch1 = Net_branch()
Cty_branch0.to(device)
Cty_branch1.to(device)
Cty_trunk = Net_trunk()
Cty_trunk.to(device)

In [ ]:
mse_cost_function = torch.nn.MSELoss() # Mean squared error
optimizer_HJ_branch0 = torch.optim.Adam(HJ_branch0.parameters(), lr=0.00001) #, weight_decay=0.0001)
optimizer_HJ_branch1 = torch.optim.Adam(HJ_branch1.parameters(), lr=0.00001) #, weight_decay=0.0001)
optimizer_HJ_trunk = torch.optim.Adam(HJ_trunk.parameters(), lr=0.00001) #, weight_decay=0.0001)
optimizer_Cty_branch0 = torch.optim.Adam(Cty_branch0.parameters(), lr=0.00001) #, weight_decay=0.0001)
optimizer_Cty_branch1 = torch.optim.Adam(Cty_branch1.parameters(), lr=0.00001) #, weight_decay=0.0001)
optimizer_Cty_trunk = torch.optim.Adam(Cty_trunk.parameters(), lr=0.00001) #, weight_decay=0.0001)

In [ ]:
# Declare univariate normal density value function
def h(x, mu, sigma):
    Y = np.array([[x]])
    X = Y.squeeze()
    return (1/(sigma*(2*math.pi)**(1/2)))*(math.e)**((-1/2)*((x-mu)**2)/sigma**2)

In [ ]:
N = 50 # mesh size
M = 200 # number of initial conditions

x_max = 7
x = np.linspace(0,x_max,N)
c = 1/8

num_u0_mixtures = 8
num_u1_mixtures = 8
k = num_u0_mixtures + num_u1_mixtures

u0_vector = np.zeros(shape=(M*N,1))
u1_vector = np.zeros(shape=(M*N,1))
x_vector = np.zeros(shape=(M*N,1))
u0_repeat_tensor = torch.zeros(size=(M*N, N))
u1_repeat_tensor = torch.zeros(size=(M*N, N))


for m in range(M):
  means = np.random.uniform(low=2.0, high=5.0, size=(k))
  sigma = np.random.uniform(low=0.5, high=0.6, size=(k))


  u0_base = np.zeros(shape=(N))
  u1_base = np.zeros(shape=(N))
  u0_base_vector = np.zeros(shape=(N,1))
  u1_base_vector = np.zeros(shape=(N,1))

  i = 0
  for j in range(N):
    u0_base[j] = c*h(x[j], means[0], sigma[0])  + c*h(x[j], means[1], sigma[1]) + \
                      c*h(x[j], means[2], sigma[2])  + c*h(x[j], means[3], sigma[3]) + \
                      c*h(x[j], means[4], sigma[4])  + c*h(x[j], means[5], sigma[5]) + \
                      c*h(x[j], means[6], sigma[6])  + c*h(x[j], means[7], sigma[7])   

    u1_base[j] = c*h(x[j], means[8], sigma[8]) + c*h(x[j], means[9], sigma[9]) + \
                  c*h(x[j], means[10], sigma[10]) + c*h(x[j], means[11], sigma[11]) + \
                  c*h(x[j], means[12], sigma[12]) + c*h(x[j], means[13], sigma[13]) + \
                  c*h(x[j], means[14], sigma[14]) + c*h(x[j], means[15], sigma[15]) 

    x_vector[m*N + i,0] = x[j]
    u0_vector[m*N + i,0] = u0_base[j]
    u1_vector[m*N + i,0] = u1_base[j]
    u0_base_vector[i,0] = u0_base[j]
    u1_base_vector[i,0] = u1_base[j]
    i += 1

  for k in range(N):
    u0_repeat_tensor[m*N+k,0:N] = Variable(torch.from_numpy(u0_base_vector[:,0]).float(), requires_grad=False).to(device)
    u1_repeat_tensor[m*N+k,0:N] = Variable(torch.from_numpy(u1_base_vector[:,0]).float(), requires_grad=False).to(device)


u0_vector_tensor = Variable(torch.from_numpy(u0_vector).float(), requires_grad=False).to(device)
u1_vector_tensor = Variable(torch.from_numpy(u1_vector).float(), requires_grad=False).to(device)
u0_repeat_tensor = Variable((u0_repeat_tensor).float(), requires_grad=False).to(device)
u1_repeat_tensor = Variable((u1_repeat_tensor).float(), requires_grad=False).to(device)
x_tensor = Variable(torch.from_numpy(x_vector).float(), requires_grad=True).to(device)

In [ ]:
def physics_informed_loss(x, t, u0, u1, Cty_trunk, Cty_branch0, Cty_branch1, HJ_trunk, HJ_branch0, HJ_branch1,  batch_size):
    
    Cty_output_trunk = Cty_trunk(x, t)
    Cty_output_branch0 = Cty_branch0(u0)
    Cty_output_branch1 = Cty_branch1(u1)   
    

    Cty_t = torch.squeeze(jacfwd(Cty_trunk, argnums=1)(x,t),3)
    Cty_x = torch.squeeze(jacfwd(Cty_trunk)(x,t),3)
    
    idx = torch.arange(batch_size)
    Cty_t = Cty_t[torch.arange(Cty_t.size(0)),:,idx] 
    Cty_x = Cty_x[torch.arange(Cty_x.size(0)),:,idx]
    
    Cty_output = torch.sum( (Cty_output_branch0*Cty_output_branch1) * Cty_output_trunk, dim=-1).unsqueeze(1)
    Cty_output_t = torch.sum( (Cty_output_branch0*Cty_output_branch1) * Cty_t, dim=-1).unsqueeze(1)
    Cty_output_x = torch.sum( (Cty_output_branch0*Cty_output_branch1) * Cty_x, dim=-1).unsqueeze(1)
    
   

    HJ_output_trunk = HJ_trunk(x, t)
    HJ_output_branch0 = HJ_branch0(u0)
    HJ_output_branch1 = HJ_branch1(u1) 
    
    HJ_t = torch.squeeze(jacfwd(HJ_trunk, argnums=1)(x,t),3)
    HJ_x = torch.squeeze(jacfwd(HJ_trunk)(x,t),3)
    HJ_xx = torch.squeeze(jacfwd(jacfwd(HJ_trunk))(x,t),5)
    
    HJ_t = HJ_t[torch.arange(HJ_t.size(0)),:,idx]  
    HJ_x = HJ_x[torch.arange(HJ_x.size(0)),:,idx]
    HJ_xx = HJ_xx[torch.arange(HJ_xx.size(0)),:,idx]
    HJ_xx = torch.squeeze(HJ_xx, 2)
    HJ_xx = HJ_xx[torch.arange(HJ_xx.size(0)),:,idx]
   
    HJ_output_t = torch.sum( (HJ_output_branch0*HJ_output_branch1) * HJ_t, dim=-1).unsqueeze(1)
    HJ_output_x = torch.sum( (HJ_output_branch0*HJ_output_branch1) * HJ_x, dim=-1).unsqueeze(1)
    HJ_output_xx = torch.sum( (HJ_output_branch0*HJ_output_branch1) * HJ_xx, dim=-1).unsqueeze(1)

    pde_residual = (Cty_output_t + Cty_output_x*HJ_output_x + Cty_output*HJ_output_xx)**2 \
                   + (HJ_output_t + (1/2)*(HJ_output_x)**2 )**2
    
    return(pde_residual)

In [ ]:
def l1_loss(x,y):
  return( (1/x.size(dim=0))*torch.sum( abs(x - y)))

In [ ]:
epochs = 10000
batch_size = 100
batches = 100

for epoch in range(epochs):
    
    t0_tensor = Variable(torch.from_numpy(np.zeros((batch_size,1))).float(), requires_grad=True).to(device)
    t1_tensor = Variable(torch.from_numpy(np.ones((batch_size,1))).float(), requires_grad=True).to(device)
    
    list_for_indices = range(0,N*M)
    
    for batch in range(batches):
        
        indices = random.sample(list_for_indices, k = batch_size)
        x_batch = Variable(torch.from_numpy(np.random.uniform(0,x_max,size=(batch_size,1))).float(), requires_grad=True).to(device)
        t_batch = Variable(torch.from_numpy(np.random.uniform(0,1,size=(batch_size,1))).float(), requires_grad=True).to(device)
        
        x_vector_batch = x_tensor[indices,]
        u0_repeat_batch = u0_repeat_tensor[indices,]
        u0_vector_batch = u0_vector_tensor[indices,]
        u1_repeat_batch = u1_repeat_tensor[indices,]
        u1_vector_batch = u1_vector_tensor[indices,]
        
        NN_branch0 = Cty_branch0(u0_repeat_batch)
        NN_branch1 = Cty_branch1(u1_repeat_batch)
        NN_trunk_boundary0 = Cty_trunk.forward(x_vector_batch, t0_tensor)
        NN_trunk_boundary1 = Cty_trunk.forward(x_vector_batch, t1_tensor)
        NN_output_boundary0 = torch.sum( (NN_branch0*NN_branch1) * NN_trunk_boundary0, dim=-1).unsqueeze(1)
        NN_output_boundary1 = torch.sum( (NN_branch0*NN_branch1) * NN_trunk_boundary1, dim=-1).unsqueeze(1)
        boundary_condition0 = mse_cost_function(NN_output_boundary0, u0_vector_batch)
        boundary_condition1 = mse_cost_function(NN_output_boundary1, u1_vector_batch)
        
        
        PINN_output = physics_informed_loss(x_batch, t_batch, u0_repeat_batch, u1_repeat_batch, \
                                            Cty_trunk, Cty_branch0, Cty_branch1, \
                                            HJ_trunk, HJ_branch0, HJ_branch1, batch_size)
     #   PINN_loss = torch.max(PINN_output)
        PINN_loss =  l1_loss(PINN_output, t0_tensor)
        
  
        loss = boundary_condition0 + boundary_condition1 + (1**(-1))*PINN_loss
  
        
        
        optimizer_Cty_trunk.zero_grad()
        optimizer_Cty_branch0.zero_grad()
        optimizer_Cty_branch1.zero_grad()
        optimizer_HJ_trunk.zero_grad()
        optimizer_HJ_branch0.zero_grad()
        optimizer_HJ_branch1.zero_grad()
        

        loss.backward()


        optimizer_Cty_trunk.step()
        optimizer_Cty_branch0.step()
        optimizer_Cty_branch1.step()
        optimizer_HJ_trunk.step()
        optimizer_HJ_branch0.step()
        optimizer_HJ_branch1.step()

        if batch % 10 == 0:
          print("Training loss:", '{:.4e}'.format(loss.data), '{:.4e}'.format(PINN_loss))

        